# PySpark Quickstart

Notebook mẫu để làm quen PySpark trên cụm.

- **Chế độ cụm**: `master = spark://spark-master:7077` — job chạy trên 2 spark-worker, xem ở http://localhost:8080
- **Chế độ local**: `master = local[*]` — Spark chạy ngay trong container Jupyter, không cần worker

Spark Application UI khi có job chạy: http://localhost:4040

## 1. Tạo SparkSession — nối vào cụm standalone

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("jupyter-quickstart")
    .master("spark://spark-master:7077")   # đổi thành local[*] để chạy local
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark

## 2. DataFrame cơ bản

In [ ]:
df = spark.createDataFrame(
    [("alice", 34), ("bob", 45), ("carol", 29), ("dave", 51)],
    ["name", "age"],
)
df.show()
df.printSchema()

In [ ]:
from pyspark.sql import functions as F

df.filter(F.col("age") > 30).groupBy().agg(F.avg("age").alias("avg_age")).show()

## 3. Ghi / đọc HDFS

`fs.defaultFS` đã trỏ sẵn về `hdfs://hadoop-namenode:9000` (trong `spark-defaults.conf`), nên đường dẫn tương đối `/user/...` chính là trên HDFS.

In [ ]:
df.write.mode("overwrite").parquet("/user/jupyter/people")

back = spark.read.parquet("/user/jupyter/people")
back.show()

Kiểm tra file trên HDFS bằng terminal:
```bash
docker exec hadoop-namenode hdfs dfs -ls /user/jupyter/people
```
hoặc mở http://localhost:9870 → Utilities → Browse the file system.

## 4. Đọc từ Kafka (streaming)

Cần package `spark-sql-kafka` — Spark sẽ tự tải lần đầu (cần mạng).

In [ ]:
# Ví dụ đọc batch từ topic 'demo' (tạo topic + gửi message trước, xem GUIDE.md 6.4)
kafka_df = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "demo")
    .option("startingOffsets", "earliest")
    .load()
)
kafka_df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)", "topic", "partition", "offset").show(truncate=False)

## 5. Dọn dẹp

In [ ]:
spark.stop()